# 02_沙箱 (Sandbox)

**来源:** [LangChain Docs — Sandbox Frontend](https://docs.langchain.com/deep-agents/frontend/sandbox)

构建 IDE 风格的前端界面，为编码智能体提供沙箱环境——包含文件浏览器、代码查看器和差异面板。编码智能体不仅需要聊天窗口，还需要像 IDE 一样的体验。

## 1. 架构

沙箱模式包含三层：

| 层 | 说明 |
|------|------|
| 深度智能体 + 沙箱后端 | 智能体自动获得文件系统工具（`read_file`、`write_file`、`edit_file`、`execute`） |
| 自定义 API 服务器 | FastAPI 应用，提供文件浏览端点供前端调用 |
| IDE 前端 | 三栏布局（文件树、代码/差异查看器、聊天），实时同步文件变更 |

```
graph LR
  UI["IDE Frontend"]
  API["API Server"]
  AGENT["createDeepAgent()"]
  SANDBOX["Sandbox"]

  UI --"useStream()"--> AGENT
  UI --"/api/sandbox/:threadId/*"--> API
  AGENT --"read/write/execute"--> SANDBOX
  API --"ls / read"--> SANDBOX
```

## 2. 沙箱生命周期与作用域

| 作用域 | 说明 | 适用场景 |
|--------|------|----------|
| **线程作用域（推荐）** | 每个 LangGraph 线程获得独立沙箱，沙箱 ID 存储在线程元数据中 | 隔离对话、持久化沙箱状态、易清理 |
| 智能体作用域 | 同一助手的所有线程共享一个沙箱 | 持久化项目环境 |
| 用户作用域 | 每个用户跨所有线程共享一个沙箱 | 需自定义认证 |
| 会话作用域 | 前端生成会话 ID，不持久化 | 演示或原型开发 |

### 线程作用域沙箱（推荐）

每个 LangGraph 线程获得自己的沙箱，沙箱 ID 存储在线程元数据中，运行时通过 `getConfig()` 解析。

In [ ]:
# 依赖安装
# pip install deepagents fastapi uvicorn

from deepagents import create_deep_agent
from deepagents.sandbox import LangSmithSandbox  # 或其他沙箱提供商

# 选择沙箱提供商
# sandbox = LangSmithSandbox.create()
# agent = create_deep_agent(model="google_genai:gemini-3.5-flash", backend=sandbox)

print("深度智能体通过 backend 参数自动获得文件系统工具和执行工具")

## 3. 按线程解析沙箱

避免在模块级别创建沙箱（会导致所有线程共享），而是在运行时按线程解析。沙箱通过 `getConfig()` 读取 `thread_id`。

In [ ]:
from deepagents import create_deep_agent
from deepagents.sandbox import LangSmithSandbox
from langgraph.config import get_config


def get_or_create_sandbox_for_thread(thread_id: str) -> LangSmithSandbox:
    """根据 thread_id 查找或创建沙箱"""
    # 实际实现需要连接 LangSmith API
    # ...
    return LangSmithSandbox()  # 占位


# sandbox = LangSmithSandbox(
#     resolve=lambda: get_or_create_sandbox_for_thread(
#         get_config()["configurable"]["thread_id"]
#     ),
# )
#
# agent = create_deep_agent(
#     model="google_genai:gemini-3.5-flash",
#     backend=sandbox,
# )

print("沙箱按线程解析，确保每个对话有独立的隔离环境")

## 4. 种子沙箱

在智能体运行之前，用项目文件填充沙箱。使用 `uploadFiles` 上传种子文件。

In [ ]:
# 前端代码（TypeScript）
# const SEED_FILES: Record<string, string> = {
#   "package.json": JSON.stringify({ name: "my-app", version: "1.0.0" }, null, 2),
#   "src/index.js": 'console.log("Hello");',
# };
#
# const encoder = new TextEncoder();
# await sandbox.uploadFiles(
#   Object.entries(SEED_FILES).map(([path, content]) => [`/app/${path}`, encoder.encode(content)]),
# );

print("种子文件上传后，可执行 sandbox.execute('cd /app && npm install') 安装依赖")

## 5. 文件浏览 API

智能体可以读写文件，但前端也需要直接访问沙箱文件系统。添加自定义 FastAPI API 服务器，并通过 `http.app` 字段暴露。

In [ ]:
# src/api/server.py
from fastapi import FastAPI, Query, Path

app = FastAPI()


@app.get("/api/sandbox/{thread_id}/tree")
async def list_tree(
    thread_id: str = Path(...),
    path: str = Query("/app"),
):
    """列出沙箱文件树"""
    # sandbox = await get_or_create_sandbox_for_thread(thread_id)
    # result = await sandbox.aexecute(f"find {path} -printf '%y\\t%s\\t%p\\n' 2>/dev/null | sort")
    # ... 解析结果
    return {"path": path, "entries": [], "sandbox_id": "placeholder"}


@app.get("/api/sandbox/{thread_id}/file")
async def read_file(
    thread_id: str = Path(...),
    path: str = Query(...),
):
    """读取沙箱中的文件内容"""
    # sandbox = await get_or_create_sandbox_for_thread(thread_id)
    # results = await sandbox.adownload_files([path])
    # return {"path": path, "content": results[0].content.decode()}
    return {"path": path, "content": ""}

print("API 服务器定义完成，需配置 langgraph.json 的 http.app 字段")

## 6. 配置 langgraph.json

在 `langgraph.json` 中注册智能体图和 API 服务器。

In [ ]:
# langgraph.json
# {
#   "graphs": {
#     "coding_agent": "./src/agents/my_agent.py:agent"
#   },
#   "env": ".env",
#   "http": {
#     "app": "./src/api/server.py:app"
#   }
# }

print("http.app 字段告诉 LangGraph 平台在默认路由之外提供自定义路由")

## 7. 构建前端 IDE

前端有三栏：文件树侧栏、代码/差异查看器和聊天面板。

### 线程创建

页面加载时创建 LangGraph 线程，将 ID 持久化到 sessionStorage，页面重载时重连同一沙箱。

In [ ]:
# React 前端代码（TypeScript）：
#
# const THREAD_KEY = "sandbox-thread-id";
# const AGENT_URL = "http://localhost:2024";
#
# function IDEPreview() {
#   const [threadId, setThreadId] = useState<string | null>(
#     () => sessionStorage.getItem(THREAD_KEY),
#   );
#
#   const updateThreadId = useCallback((id: string | null) => {
#     setThreadId(id);
#     if (id) sessionStorage.setItem(THREAD_KEY, id);
#     else sessionStorage.removeItem(THREAD_KEY);
#   }, []);
#
#   const stream = useStream<typeof myAgent>({
#     apiUrl: AGENT_URL,
#     assistantId: "coding_agent",
#     threadId,
#     onThreadId: updateThreadId,
#   });
# }

print("前端 IDE 布局：文件树(208px) + 代码/差异(弹性) + 聊天(320px)")

### 实时文件同步

关键：在智能体工作过程中实时更新文件，而不是等运行结束。监控流中的 `ToolMessage` 实例：

In [ ]:
# React 实时文件同步示例：
#
# const FILE_MUTATING_TOOLS = new Set(["write_file", "edit_file", "execute"]);
#
# useEffect(() => {
#   // 从 AI 消息中构建文件变更工具调用映射
#   const toolCallMap = new Map();
#   for (const msg of stream.messages) {
#     if (!AIMessage.isInstance(msg)) continue;
#     for (const tc of msg.tool_calls ?? []) {
#       if (tc.id && FILE_MUTATING_TOOLS.has(tc.name)) {
#         toolCallMap.set(tc.id, { name: tc.name, args: tc.args });
#       }
#     }
#   }
#   // 当 ToolMessage 出现时，刷新对应文件
#   for (const msg of stream.messages) {
#     if (!ToolMessage.isInstance(msg)) continue;
#     const call = toolCallMap.get(msg.tool_call_id);
#     if (!call) continue;
#     if (call.name === "write_file" || call.name === "edit_file") {
#       refreshSingleFile(call.args.path);
#     } else if (call.name === "execute") {
#       refreshAllFiles();
#     }
#   }
# }, [stream.messages]);

print("实时同步：write_file/edit_file 刷新单个文件，execute 刷新全部")

### 差异检测与展示

在智能体每次运行前快照文件内容，刷新后比较识别变更文件。

| 框架 | 差异库 | 组件 |
|------|--------|------|
| React | `@pierre/diffs` | `<FileDiff>` + `parseDiffFromFile` |
| Vue | `@git-diff-view/vue` | `<DiffView>` + `generateDiffFile` |
| Svelte | `@git-diff-view/svelte` | `<DiffView>` + `generateDiffFile` |
| Angular | `ngx-diff` | `<ngx-unified-diff>` |

## 8. 最佳实践

| 实践 | 说明 |
|------|------|
| 使用线程作用域沙箱 | 生产环境推荐，沙箱 ID 存储在元数据中，运行时通过 `getConfig()` 解析 |
| 共享 `getOrCreateSandboxForThread` | 智能体后端和 API 服务器使用同一函数解析沙箱 |
| 持久化 threadId | 使用 sessionStorage，页面重载重连同一沙箱 |
| 每次相关工具调用时同步文件 | 不等到运行结束，使 IDE 感觉实时 |
| 变更文件默认差异视图 | 用户点击被修改的文件时先显示 diff |
| 过滤 node_modules | 文件树中过滤掉依赖目录 |

---

**参考链接**
- [LangChain Docs — Sandbox Pattern](https://docs.langchain.com/deep-agents/frontend/sandbox)
- [LangSmith Sandbox](https://docs.smith.langchain.com/tracing/faq/langsmith-sandbox)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)